# Moisture sensor → as-fed nutrients

The moisture probe measures **water %**, not protein. Tables are % of dry matter.

`as_fed = value_DM × (1 − moisture/100)`

Data: BIS IS 2052:2023 moisture max 11% for compounded feed; NDDB typical moistures.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../..").resolve()
if not (ROOT / "smartfeed").exists():
    ROOT = Path(".").resolve()
    while ROOT != ROOT.parent and not (ROOT / "smartfeed").exists():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"
print("ROOT", ROOT)
print("DATA", DATA)


ROOT /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111
DATA /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111/data/processed


In [2]:
from smartfeed.moisture import as_fed, assess_moisture, dry_matter_pct

bis = pd.read_csv(DATA / "bis_is2052_2023_compounded_feed_limits.csv")
print(bis[bis.characteristic == "Moisture"])

print("10% moisture, table CP 35% DM -> as-fed", as_fed(35, 10))
print(json.dumps(assess_moisture(10, form="ingredient"), indent=2))
print(json.dumps(assess_moisture(16, form="compounded"), indent=2))
print(json.dumps(assess_moisture(70, form="silage"), indent=2))
print("DM from 70% moisture", dry_matter_pct(70))


  characteristic           unit  type_I  type_II   basis
0       Moisture  % as-fed, max    11.0     11.0  as_fed
10% moisture, table CP 35% DM -> as-fed 31.5
{
  "present": true,
  "moisture_pct": 10.0,
  "dry_matter_pct": 90.0,
  "flags": [],
  "note": "Moisture converts % DM to as-fed. It is not crude protein."
}
{
  "present": true,
  "moisture_pct": 16.0,
  "dry_matter_pct": 84.0,
  "flags": [
    "dry_feed_above_safe_store_moisture",
    "above_bis_moisture_max_11"
  ],
  "note": "Moisture converts % DM to as-fed. It is not crude protein."
}
{
  "present": true,
  "moisture_pct": 70.0,
  "dry_matter_pct": 30.0,
  "flags": [
    "silage_dm_in_typical_band"
  ],
  "note": "Moisture converts % DM to as-fed. It is not crude protein."
}
DM from 70% moisture 30.0


In [3]:
art = Path("artifacts")
art.mkdir(exist_ok=True)
(art / "metrics.json").write_text(
    json.dumps(
        {
            "model": "as_fed_identity",
            "formula": "value_asfed = value_dm * (1 - moisture/100)",
            "bis_compounded_moisture_max": 11,
            "measures_protein": False,
        },
        indent=2,
    )
)
print("saved", art / "metrics.json")


saved artifacts/metrics.json
